# GPU Matrix Multiplication Benchmark

## Setup: check the GPU and libraries


In [55]:
!nvidia-smi

Thu Oct  8 00:19:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             26W /   70W |     839MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [56]:
import numpy as np
import cupy as cp
from numba import cuda

print("NumPy", np.__version__, "| CuPy", cp.__version__)
print("Numba sees a GPU:", cuda.is_available())
print("GPU:", cuda.get_current_device().name)

NumPy 2.1.3 | CuPy 14.0.1
Numba sees a GPU: True
GPU: Tesla T4


In [57]:
import time

rng = np.random.default_rng(seed=0)

def make_matrices(n):
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    return A, B

## Section 1: Pure Python Triple Loop

In [58]:
#matrix multiplication (the 3 loops show how this becomes n^3)
def matmul_python(A, B):
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            total = 0.0
            for k in range(n):
                total += A[i][k] * B[k][j]
            C[i][j] = total
    return C

In [59]:
A, B = make_matrices(64)
C_python = matmul_python(A.tolist(), B.tolist())
C_numpy = A @ B
print("Matches NumPy:", np.allclose(C_python, C_numpy, rtol=1e-4))

Matches NumPy: True


In [60]:
#doubling n should take about 8x longer as the loop is O(n^3)
for n in [32, 64, 128, 256]:
    A, B = make_matrices(n)
    A_list, B_list = A.tolist(), B.tolist()
    start = time.perf_counter()
    matmul_python(A_list, B_list)
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.3f} seconds")


n =   32: 0.003 seconds
n =   64: 0.016 seconds
n =  128: 0.133 seconds
n =  256: 1.206 seconds


## Section 2: NumPy (CPU, optimized BLAS)

In [61]:
#NumPy (CPU, optimized BLAS)
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    start = time.perf_counter()
    A @ B
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000038 seconds
n =   64: 0.000036 seconds
n =  128: 0.001396 seconds
n =  256: 0.000445 seconds
n =  512: 0.003183 seconds
n = 1024: 0.021569 seconds
n = 2048: 0.142301 seconds
n = 4096: 1.310479 seconds


In [62]:
# comparing the speed of NumPy and python
A, B = make_matrices(256)
A_list, B_list = A.tolist(), B.tolist()

start = time.perf_counter()
matmul_python(A_list, B_list)
python_time = time.perf_counter() - start

start = time.perf_counter()
A @ B
numpy_time = time.perf_counter() - start

print(f"Python: {python_time:.3f} s | NumPy: {numpy_time:.6f} s | Speedup: {python_time / numpy_time:,.0f}x")

Python: 1.913 s | NumPy: 0.002563 s | Speedup: 747x


## Section 3: CuPy (NVIDIA GPU via cuBLAS)

In [63]:
# using CuPy instead by copying matrices to GPU memory to multiply,
# then copying the result back to check it
A, B = make_matrices(512)
A_gpu = cp.asarray(A)
B_gpu = cp.asarray(B)
C_gpu = A_gpu @ B_gpu
print("Matches NumPy:", np.allclose(cp.asnumpy(C_gpu), A @ B, rtol=1e-4))

Matches NumPy: True


In [64]:
# GPU calls return immediately, so timing without synchronize is wrong
A, B = make_matrices(4096)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
A_gpu @ B_gpu
cp.cuda.Device().synchronize()

start = time.perf_counter()
A_gpu @ B_gpu
no_sync = time.perf_counter() - start

start = time.perf_counter()
A_gpu @ B_gpu
cp.cuda.Device().synchronize()
with_sync = time.perf_counter() - start

print(f"Without synchronize: {no_sync:.6f} s  wrong b/c only measures sending the job")
print(f"With synchronize:    {with_sync:.6f} s  correct timing")

Without synchronize: 0.000216 s  wrong b/c only measures sending the job
With synchronize:    0.088421 s  correct timing


In [65]:
# The first GPU call includes one-time setup, so it's much slower than later calls
A, B = make_matrices(1024)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
for attempt in range(1, 4):
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    print(f"Call {attempt}: {time.perf_counter() - start:.6f} s")

Call 1: 0.000712 s
Call 2: 0.000674 s
Call 3: 0.000613 s


In [66]:
# GPU compute time only (data already on the GPU), with warm-up and synchronize
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
    A_gpu @ B_gpu                      # warm-up for this size
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000213 seconds
n =   64: 0.000111 seconds
n =  128: 0.000094 seconds
n =  256: 0.000077 seconds
n =  512: 0.000120 seconds
n = 1024: 0.000557 seconds
n = 2048: 0.003740 seconds
n = 4096: 0.029014 seconds


## Section 4: Custom CUDA kernel (Numba)

In [67]:
#custom CUDA kernel (Numba)
from numba import float32

#each GPU thread computes an entry of C with its location stored
#this is why we don't need the i and j loops from the python example
#x = column, so the 32 threads in a warp read neighboring memory (coalesced)
@cuda.jit
def matmul_kernel(A, B, C):
    col, row = cuda.grid(2)               # this thread's position in the grid
    n = C.shape[0]
    if row < n and col < n:               # skip threads that fall off the edge
        total = float32(0.0)
        for k in range(n):
            total += A[row, k] * B[k, col]
        C[row, col] = total

In [68]:
#copy inputs to the GPU, creating an empty output, and launch one thread per entry of C
n = 512
A, B = make_matrices(n)
A_dev = cuda.to_device(A)
B_dev = cuda.to_device(B)
C_dev = cuda.device_array((n, n), dtype=np.float32)

threads_per_block = (16, 16)
blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
cuda.synchronize()

print("Matches NumPy:", np.allclose(C_dev.copy_to_host(), A @ B, rtol=1e-4))

Matches NumPy: True


In [69]:
import warnings
warnings.filterwarnings("ignore", message="Grid size")   # small grids underuse the GPU; expected here

#kernel compute time only, with warm-up
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array((n, n), dtype=np.float32)
    blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

    matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)   # warm-up
    cuda.synchronize()

    start = time.perf_counter()
    matmul_kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
    cuda.synchronize()
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000086 seconds
n =   64: 0.000081 seconds
n =  128: 0.000102 seconds
n =  256: 0.000259 seconds
n =  512: 0.001365 seconds
n = 1024: 0.009997 seconds
n = 2048: 0.078711 seconds
n = 4096: 0.627296 seconds


### Testing thread mapping and memory coalescing

Same math, but seeing how threads handle consecutive rows instead of columns.

In [70]:
# Slower version for comparison: x = row, so a warp reads down a column of A (scattered in memory)
@cuda.jit
def matmul_kernel_uncoalesced(A, B, C):
    row, col = cuda.grid(2)
    n = C.shape[0]
    if row < n and col < n:
        total = float32(0.0)
        for k in range(n):
            total += A[row, k] * B[k, col]
        C[row, col] = total

In [71]:
# same math, different thread-to-entry mapping
for n in [2048, 4096]:
    A, B = make_matrices(n)
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array((n, n), dtype=np.float32)
    blocks_per_grid = ((n + 15) // 16, (n + 15) // 16)

    for name, kernel in [("uncoalesced", matmul_kernel_uncoalesced), ("coalesced", matmul_kernel)]:
        kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)   # warm-up
        cuda.synchronize()
        start = time.perf_counter()
        kernel[blocks_per_grid, threads_per_block](A_dev, B_dev, C_dev)
        cuda.synchronize()
        print(f"n = {n}, {name:11}: {time.perf_counter() - start:.6f} seconds")

print("Coalesced matches NumPy:", np.allclose(C_dev.copy_to_host(), A @ B, rtol=1e-4))

n = 2048, uncoalesced: 0.154726 seconds
n = 2048, coalesced  : 0.080858 seconds
n = 4096, uncoalesced: 1.272070 seconds
n = 4096, coalesced  : 0.632085 seconds
Coalesced matches NumPy: True


## Section 5: Benchmark (median of 5 runs, with and without data transfer)

In [72]:
import statistics

# warm up then time `repeats` runs and return the median
def benchmark(func, repeats=5):
    func()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func()
        times.append(time.perf_counter() - start)
    return statistics.median(times)

In [73]:
def cupy_compute(A_gpu, B_gpu):          # data already on the GPU
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()

def cupy_with_transfer(A, B):            # copy in, multiply, copy the result back
    C_gpu = cp.asarray(A) @ cp.asarray(B)
    return cp.asnumpy(C_gpu)             # copying back waits for the GPU to finish

def kernel_compute(A_dev, B_dev, C_dev, blocks):
    matmul_kernel[blocks, threads_per_block](A_dev, B_dev, C_dev)
    cuda.synchronize()

def kernel_with_transfer(A, B, blocks):
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array(A.shape, dtype=np.float32)
    matmul_kernel[blocks, threads_per_block](A_dev, B_dev, C_dev)
    return C_dev.copy_to_host()

In [74]:
import pandas as pd

sizes = [32, 64, 128, 256, 512, 1024, 2048, 4096]
results = {"Python": [], "NumPy": [], "CuPy": [], "CuPy + transfer": [],
           "Kernel": [], "Kernel + transfer": []}

for n in sizes:
    A, B = make_matrices(n)
    A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
    A_dev, B_dev = cuda.to_device(A), cuda.to_device(B)
    C_dev = cuda.device_array((n, n), dtype=np.float32)
    blocks = ((n + 15) // 16, (n + 15) // 16)

    # The Python loop would take hours at large sizes, so stop at 256
    if n <= 256:
        A_list, B_list = A.tolist(), B.tolist()
        results["Python"].append(benchmark(lambda: matmul_python(A_list, B_list)))
    else:
        results["Python"].append(np.nan)

    results["NumPy"].append(benchmark(lambda: A @ B))
    results["CuPy"].append(benchmark(lambda: cupy_compute(A_gpu, B_gpu)))
    results["CuPy + transfer"].append(benchmark(lambda: cupy_with_transfer(A, B)))
    results["Kernel"].append(benchmark(lambda: kernel_compute(A_dev, B_dev, C_dev, blocks)))
    results["Kernel + transfer"].append(benchmark(lambda: kernel_with_transfer(A, B, blocks)))
    print(f"n = {n} done")

df = pd.DataFrame(results, index=sizes)
df.index.name = "n"
(df * 1000).round(3)        # show times in milliseconds

n = 32 done
n = 64 done
n = 128 done
n = 256 done
n = 512 done
n = 1024 done
n = 2048 done
n = 4096 done


,Python,NumPy,CuPy,CuPy + transfer,Kernel,Kernel + transfer
n,,,,,,
32,1.883,0.004,0.037,0.176,0.051,0.663
64,15.489,0.008,0.038,0.178,0.051,0.605
128,129.870,0.065,0.055,0.235,0.107,0.866
256,1203.627,0.361,0.099,0.380,0.379,1.374
512,NaN,2.610,0.180,0.897,2.331,4.088
1024,NaN,17.033,0.883,3.181,17.843,18.686
2048,NaN,136.063,3.212,13.581,79.136,86.545
4096,NaN,1308.471,47.608,74.941,633.900,687.471


In [75]:
# Speedup = slower time / faster time
speedup = pd.DataFrame({
    "CuPy vs NumPy": df["NumPy"] / df["CuPy"],
    "CuPy + transfer vs NumPy": df["NumPy"] / df["CuPy + transfer"],
    "Kernel vs NumPy": df["NumPy"] / df["Kernel"],
    "CuPy vs Python": df["Python"] / df["CuPy"],
})
speedup.round(1)

,CuPy vs NumPy,CuPy + transfer vs NumPy,Kernel vs NumPy,CuPy vs Python
n,,,,
32,0.1,0.0,0.1,50.8
64,0.2,0.0,0.2,406.6
128,1.2,0.3,0.6,2346.6
256,3.7,0.9,1.0,12212.0
512,14.5,2.9,1.1,NaN
1024,19.3,5.4,1.0,NaN
2048,42.4,10.0,1.7,NaN
4096,27.5,17.5,2.1,NaN
